# 04. End-to-End Planner Flow
이 노트북은 Planner부터 DB/RAG/최종 답변까지 한 번에 검증하는 테스트용 노트북입니다.
노트북은 서비스 호출과 결과 확인만 담당하고, 실제 로직은 `.py` 서비스(`planner_flow_runner.py`)에 둡니다.
`query_type`에 따라 어떤 경로가 실행되는지 단계별로 확인할 수 있습니다.

In [15]:
from pathlib import Path
import sys
import traceback
from pprint import pprint

import pandas as pd
from IPython.display import display

WORKDIR = Path.cwd()
REPO_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR
if REPO_ROOT.name == "Root_Stream":
    REPO_ROOT = REPO_ROOT.parent
ROOT_STREAM_PATH = REPO_ROOT / "Root_Stream"

for path_item in (REPO_ROOT, ROOT_STREAM_PATH):
    if str(path_item) not in sys.path:
        sys.path.append(str(path_item))

MODULE_IMPORT_ERRORS = {}

try:
    from Root_Stream.services.e2e.planner_flow_runner import (
        run_planner_step,
        run_db_step,
        summarize_db_execution_payload,
        run_doc_rag_step,
        run_final_answer_step,
        run_end_to_end_planner_flow,
    )
except Exception as error:
    MODULE_IMPORT_ERRORS["Root_Stream.services.e2e.planner_flow_runner"] = str(error)
    print("[import 실패] Root_Stream.services.e2e.planner_flow_runner")
    traceback.print_exc()

try:
    from Root_Stream.utils.config_loader import load_config
except Exception as error:
    MODULE_IMPORT_ERRORS["Root_Stream.utils.config_loader"] = str(error)
    print("[import 실패] Root_Stream.utils.config_loader")
    traceback.print_exc()

if MODULE_IMPORT_ERRORS:
    print("아래 모듈 import가 실패했습니다:")
    pprint(MODULE_IMPORT_ERRORS)
else:
    print("필수 모듈 import 성공")

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"ROOT_STREAM_PATH: {ROOT_STREAM_PATH}")


필수 모듈 import 성공
REPO_ROOT: c:\Users\김민한\Desktop\개발\DB_to_LLM
ROOT_STREAM_PATH: c:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream


## 실행 전 체크 항목
- Planner 호출 가능 여부(LLM 연결/프롬프트 키 포함)
- DB 연결 필요 여부(실DB 실행 또는 mock 모드)
- `Root_Ingest/doc_rag`, `Root_Ingest/data_rag` 준비 여부
- 일반 문서 RAG 컬렉션 `rag_chunks` 존재 여부

In [16]:
CONFIG_PATH = ROOT_STREAM_PATH / "config" / "config.yaml"
config = load_config(CONFIG_PATH)

planner_enabled = config.get("planner", {}).get("enabled", "현재 config에 명시 필드 없음")
sql_retrieval_cfg = config.get("sql_retrieval")
doc_retrieval_cfg = config.get("doc_retrieval")

summary_rows = [
    {"항목": "config_path", "값": str(CONFIG_PATH)},
    {"항목": "llm_provider", "값": config.get("llm_provider")},
    {"항목": "mode", "값": config.get("mode")},
    {"항목": "planner.enabled", "값": planner_enabled},
    {
        "항목": "sql_retrieval",
        "값": sql_retrieval_cfg if sql_retrieval_cfg is not None else "현재 config에 명시 필드 없음(기존 retrieval 기반 추론)",
    },
    {
        "항목": "doc_retrieval",
        "값": doc_retrieval_cfg if doc_retrieval_cfg is not None else "현재 config에 명시 필드 없음(기본 data_rag/rag_chunks fallback 사용)",
    },
]

display(pd.DataFrame(summary_rows))


[2026-04-10 09:47:27] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: c:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:47:27] [INFO] [config_loader.py:load_config:30] 설정 로드 완료


,항목,값
0,config_path,c:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\...
1,llm_provider,ollama
2,mode,rag_prompt_llm
3,planner.enabled,False
4,sql_retrieval,"{'enabled': True, 'embedding_model': 'sentence..."
5,doc_retrieval,"{'enabled': True, 'embedding_model': 'sentence..."


In [17]:
question = "가장 최근 발생한 에러를 찾고 어떤 의미인지 설명해줘"  # 질문 1개만 사용

USE_MOCK_DB_RESULT = True
USE_DOC_RAG = True
DOC_RAG_TOP_K = 4
SQL_GENERATION_MODE = None  # None?? planner/query_type ?? ?? ??

print("현재 실행 질문:")
print(question)
print(f"USE_MOCK_DB_RESULT={USE_MOCK_DB_RESULT}, USE_DOC_RAG={USE_DOC_RAG}, DOC_RAG_TOP_K={DOC_RAG_TOP_K}")


현재 실행 질문:
가장 최근 발생한 에러를 찾고 어떤 의미인지 설명해줘
USE_MOCK_DB_RESULT=True, USE_DOC_RAG=True, DOC_RAG_TOP_K=4


In [21]:
planner_output = None
planner_error = None
query_type = None
planner_result = None
planner_raw = None
reasoning_summary = None

try:
    planner_output = run_planner_step(question=question, config_path=CONFIG_PATH)
    planner_raw = planner_output.get("planner_raw")
    planner_result = planner_output.get("planner_result")
    query_type = planner_output.get("query_type")
    reasoning_summary = planner_output.get("reasoning_summary")

    print("[Planner Raw Result]")
    print(planner_raw)

    print("\n[Planner Parsed Result]")
    print(planner_result)

    print("\n[Planner 핵심 필드]")
    print(f"query_type: {query_type}")
    if reasoning_summary:
        print(f"reasoning_summary: {reasoning_summary}")
    else:
        print("reasoning_summary: 현재 Planner 스키마에 reasoning_summary 없음")

    steps_df = pd.DataFrame((planner_result or {}).get("steps", []))
    if not steps_df.empty:
        display(steps_df[["step", "type", "goal", "depends_on"]])
    else:
        print("steps 정보가 비어 있습니다.")
except Exception as error:
    planner_error = str(error)
    print(f"Planner 실행 실패: {planner_error}")
    traceback.print_exc()


[2026-04-10 09:48:53] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:48:53] [INFO] [config_loader.py:load_config:30] 설정 로드 완료
[2026-04-10 09:48:53] [INFO] [prompt_manager.py:_load_prompts:34] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-10 09:48:53] [INFO] [prompt_manager.py:_load_prompts:46] 프롬프트 파일 로드 완료: prompt_count=5
[2026-04-10 09:48:53] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:48:53] [INFO] [config_loader.py:load_config:30] 설정 로드 완료
[2026-04-10 09:48:53] [INFO] [prompt_manager.py:_load_prompts:34] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-10 09:48:53] [INFO] [prompt_manager.py:_load_prompts:46] 프롬프트 파일 로드 완료: prompt_count=5
[2026-04-10 09:48:53] [INFO] [planner_service.py:__init__:44] PlannerServi

[Planner Raw Result]
{
  "is_composite": true,
  "query_type": "DB_THEN_RAG",
  "steps": [
    {
      "step": 1,
      "type": "db",
      "goal": "가장 최근 발생한 에러를 찾는다.",
      "depends_on": []
    },
    {
      "step": 2,
      "type": "rag",
      "goal": "찾은 가장 최근 발생한 에러의 의미와 관련된 문서나 설명을 찾아낸다.",
      "depends_on": [
        1
      ]
    }
  ]
}

[Planner Parsed Result]
{'is_composite': True, 'query_type': 'DB_THEN_RAG', 'steps': [{'step': 1, 'type': 'db', 'goal': '가장 최근 발생한 에러를 찾는다.', 'depends_on': []}, {'step': 2, 'type': 'rag', 'goal': '찾은 가장 최근 발생한 에러의 의미와 관련된 문서나 설명을 찾아낸다.', 'depends_on': [1]}]}

[Planner 핵심 필드]
query_type: DB_THEN_RAG
reasoning_summary: 현재 Planner 스키마에 reasoning_summary 없음


,step,type,goal,depends_on
0,1,db,가장 최근 발생한 에러를 찾는다.,[]
1,2,rag,찾은 가장 최근 발생한 에러의 의미와 관련된 문서나 설명을 찾아낸다.,[1]


## Planner 결과 해석
- `DB_ONLY`: DB 조회(SQL 생성/검증/실행)만 수행
- `RAG_ONLY`: 일반 문서 RAG 검색 중심으로 답변
- `GENERAL`: DB/RAG 없이 일반 LLM 답변 중심
- `DB_THEN_RAG`: DB 결과를 얻은 뒤 문서 RAG를 추가 수행
- `DB_THEN_GENERAL`: DB 결과를 얻고 일반 설명으로 마무리
- `RAG_THEN_GENERAL`: 문서 RAG를 먼저 참고하고 일반 설명으로 마무리

In [22]:
generated_sql = None
selected_sql_mode = None
sql_validation_result = None
db_rows = []
db_result_summary = None
rag_query = None
rag_docs = []
final_answer = None
executed_steps = []
errors = []

if planner_error:
    errors.append(f"Planner 실행 실패: {planner_error}")
elif planner_output:
    executed_steps.append("planner")

print("실행 컨텍스트 초기화 완료")


실행 컨텍스트 초기화 완료


In [23]:
db_step_output = None

if query_type in {"DB_ONLY", "DB_THEN_RAG", "DB_THEN_GENERAL"}:
    try:
        db_step_output = run_db_step(
            question=question,
            query_type=query_type,
            planner_steps=(planner_result or {}).get("steps", []),
            config_path=CONFIG_PATH,
            use_mock_db_result=USE_MOCK_DB_RESULT,
            sql_generation_mode=SQL_GENERATION_MODE,
        )

        generated_sql = db_step_output.get("generated_sql")
        selected_sql_mode = db_step_output.get("selected_sql_mode")
        sql_validation_result = db_step_output.get("sql_validation_result")
        db_rows = db_step_output.get("db_rows") or []
        db_result_summary = db_step_output.get("db_result_summary")
        executed_steps.extend(db_step_output.get("executed_steps", []))
        errors.extend(db_step_output.get("errors", []))

        print(f"selected_sql_mode: {selected_sql_mode}")
        print("\n[생성 SQL]")
        print(generated_sql)

        print("\n[SQL Guard 결과]")
        pprint(sql_validation_result)

        if db_rows:
            print(f"\nDB 결과 row_count: {len(db_rows)}")
            display(pd.DataFrame(db_rows).head(20))
        else:
            print("DB 결과가 비어 있습니다.")
    except Exception as error:
        message = f"DB step 실행 중 예외: {error}"
        errors.append(message)
        print(message)
        traceback.print_exc()
else:
    print(f"query_type={query_type} 이므로 DB step을 건너뜁니다.")


[2026-04-10 09:49:06] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:49:06] [INFO] [config_loader.py:load_config:30] 설정 로드 완료
[2026-04-10 09:49:06] [INFO] [prompt_manager.py:_load_prompts:34] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-10 09:49:06] [INFO] [prompt_manager.py:_load_prompts:46] 프롬프트 파일 로드 완료: prompt_count=5
[2026-04-10 09:49:06] [INFO] [query_service.py:generate_stream_query:79] STREAM 공통 생성 실행: mode=rag_prompt_llm, config=C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:49:06] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:49:06] [INFO] [config_loader.py:load_config:30] 설정 로드 완료
[2026-04-10 09:49:06] [INFO] [prompt_manager.py:_load_prompts:34] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_te

selected_sql_mode: rag_prompt

[생성 SQL]
```sql
SELECT TOP 1 EQPID, MODULEID, EVENTTIME, UNIT, STEP, GLASSID, GLASSID2, LOTID, HOSTPPID, EQPPPID, ACTION, ALARMID, ALARMTEXT, FILENAME, CREATETIME
FROM ERROR_LOG_DATA
ORDER BY EVENTTIME DESC
```

[SQL Guard 결과]
{'error': None,
 'is_valid': True,
 'validated_sql': 'SELECT TOP 1 EQPID, MODULEID, EVENTTIME, UNIT, STEP, '
                  'GLASSID, GLASSID2, LOTID, HOSTPPID, EQPPPID, ACTION, '
                  'ALARMID, ALARMTEXT, FILENAME, CREATETIME\n'
                  'FROM ERROR_LOG_DATA\n'
                  'ORDER BY EVENTTIME DESC'}

DB 결과 row_count: 3


,eqpid,error_count,latest_event_time
0,EQP-01,12,2026-04-08 13:21:00
1,EQP-03,8,2026-04-08 12:54:10
2,EQP-07,4,2026-04-08 11:40:09


In [24]:
if db_result_summary is None and db_rows:
    # DB step 결과 요약이 비어 있으면 helper로 재생성
    db_result_summary = summarize_db_execution_payload(
        {
            "row_count": len(db_rows),
            "columns": list(db_rows[0].keys()) if db_rows else [],
            "rows": db_rows,
        }
    )

if db_result_summary:
    print("[DB 결과 요약]")
    pprint(db_result_summary)
    print("\n요약 텍스트:")
    print(db_result_summary.get("summary_text"))

    head_rows = db_result_summary.get("head_rows", [])
    if head_rows:
        display(pd.DataFrame(head_rows))
else:
    print("DB 결과 요약 정보가 없습니다.")


[DB 결과 요약]
{'columns': ['eqpid', 'error_count', 'latest_event_time'],
 'head_rows': [{'eqpid': 'EQP-01',
                'error_count': 12,
                'latest_event_time': '2026-04-08 13:21:00'},
               {'eqpid': 'EQP-03',
                'error_count': 8,
                'latest_event_time': '2026-04-08 12:54:10'},
               {'eqpid': 'EQP-07',
                'error_count': 4,
                'latest_event_time': '2026-04-08 11:40:09'}],
 'key_points': ['총 3건 조회',
                '컬럼 수 3개',
                '첫 행 예시: eqpid=EQP-01, error_count=12, '
                'latest_event_time=2026-04-08 13:21:00'],
 'row_count': 3,
 'summary_text': 'row_count: 3\n'
                 "columns: ['eqpid', 'error_count', 'latest_event_time']\n"
                 'key_points: 총 3건 조회; 컬럼 수 3개; 첫 행 예시: eqpid=EQP-01, '
                 'error_count=12, latest_event_time=2026-04-08 13:21:00'}

요약 텍스트:
row_count: 3
columns: ['eqpid', 'error_count', 'latest_event_time']
key_points: 총 3건 조회

,eqpid,error_count,latest_event_time
0,EQP-01,12,2026-04-08 13:21:00
1,EQP-03,8,2026-04-08 12:54:10
2,EQP-07,4,2026-04-08 11:40:09


In [25]:
rag_step_output = None

if query_type in {"RAG_ONLY", "DB_THEN_RAG", "RAG_THEN_GENERAL"}:
    if USE_DOC_RAG:
        try:
            rag_step_output = run_doc_rag_step(
                question=question,
                query_type=query_type,
                db_result_summary=db_result_summary,
                config_path=CONFIG_PATH,
                use_doc_rag=USE_DOC_RAG,
                doc_rag_top_k=DOC_RAG_TOP_K,
            )

            rag_query = rag_step_output.get("rag_query")
            rag_docs = rag_step_output.get("rag_docs") or []
            executed_steps.extend(rag_step_output.get("executed_steps", []))
            errors.extend(rag_step_output.get("errors", []))

            print("[RAG 질의]")
            print(rag_query)
            print("\n[RAG 설정]")
            pprint(rag_step_output.get("rag_config"))

            if rag_docs:
                display_rows = []
                for doc in rag_docs:
                    metadata = doc.get("metadata", {}) if isinstance(doc, dict) else {}
                    source_name = metadata.get("source") if isinstance(metadata, dict) else None
                    text_preview = str(doc.get("text", "")).replace("\n", " ")[:200]
                    display_rows.append(
                        {
                            "chunk_id": doc.get("chunk_id"),
                            "source": source_name,
                            "score": doc.get("score"),
                            "text_preview": text_preview,
                        }
                    )
                print(f"RAG 검색 문서 수: {len(rag_docs)}")
                display(pd.DataFrame(display_rows))
            else:
                print("RAG 검색 결과가 없습니다.")
        except Exception as error:
            message = f"문서 RAG step 실행 중 예외: {error}"
            errors.append(message)
            print(message)
            traceback.print_exc()
    else:
        print("USE_DOC_RAG=False 이므로 문서 RAG step을 건너뜁니다.")
else:
    print(f"query_type={query_type} 이므로 문서 RAG step을 건너뜁니다.")


[2026-04-10 09:49:30] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:49:30] [INFO] [config_loader.py:load_config:30] 설정 로드 완료
[2026-04-10 09:49:30] [INFO] [prompt_manager.py:_load_prompts:34] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-10 09:49:30] [INFO] [prompt_manager.py:_load_prompts:46] 프롬프트 파일 로드 완료: prompt_count=5
[2026-04-10 09:49:30] [INFO] [embedding_service.py:_get_model:69] 임베딩 모델 로드 시작: model=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
[2026-04-10 09:49:35] [INFO] [embedding_service.py:_get_model:73] 임베딩 모델 로드 완료: model=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
[2026-04-10 09:49:35] [INFO] [embedding_service.py:embed_query:97] 질의 임베딩 시작: text_length=215
[2026-04-10 09:49:35] [INFO] [chroma_retriever.py:_get_collection:79] Chroma 컬렉션 연결 시작: path=C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Ingest\data_rag\chr

[RAG 질의]
가장 최근 발생한 에러를 찾고 어떤 의미인지 설명해줘

[DB 결과 요약]
row_count: 3
columns: ['eqpid', 'error_count', 'latest_event_time']
key_points: 총 3건 조회; 컬럼 수 3개; 첫 행 예시: eqpid=EQP-01, error_count=12, latest_event_time=2026-04-08 13:21:00

[RAG 설정]
{'chroma_path': 'C:\\Users\\김민한\\Desktop\\개발\\DB_to_LLM\\Root_Ingest\\data_rag\\chroma',
 'collection_name': 'rag_chunks',
 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
 'source': 'doc_retrieval',
 'top_k': 4}
RAG 검색 문서 수: 3


,chunk_id,source,score,text_preview
0,ee887e0e0bd8ad0a_chunk_0001,None,1.349907,전 압력 추이 로그 확인 2) 배기 밸브/펌프 상태 비트 확인 3) 동일 시간대 설...
1,ee887e0e0bd8ad0a_chunk_0000,None,1.376323,최근 발생 에러 ## 1.1 S1JOA08J\_NHTU\_XB01\_EX01 *...
2,ee887e0e0bd8ad0a_chunk_0002,None,1.500453,"| **영향 범위** | 로딩/언로딩 지연, 사이클 타임 증가, 특정 구간 반복..."


In [13]:
try:
    final_out = run_final_answer_step(
        question=question,
        planner_result=planner_result,
        generated_sql=generated_sql,
        db_result_summary=db_result_summary,
        rag_docs=rag_docs,
        config_path=CONFIG_PATH,
        errors=errors,
    )
    final_answer = final_out.get("final_answer")
    errors.extend(final_out.get("errors", []))
    executed_steps.append("final_answer")

    print("[최종 한국어 답변]")
    print(final_answer)
except Exception as error:
    message = f"최종 답변 생성 단계 예외: {error}"
    errors.append(message)
    print(message)
    traceback.print_exc()


[2026-04-09 17:05:44] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-09 17:05:44] [INFO] [config_loader.py:load_config:30] 설정 로드 완료
[2026-04-09 17:05:44] [INFO] [prompt_manager.py:_load_prompts:34] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-09 17:05:44] [INFO] [prompt_manager.py:_load_prompts:46] 프롬프트 파일 로드 완료: prompt_count=5
[2026-04-09 17:05:44] [INFO] [ollama_client.py:generate:122] Ollama 호출 시작: model=qwen2.5:14b, endpoint=http://192.168.0.74:11434/api/chat
[2026-04-09 17:06:13] [INFO] [ollama_client.py:generate:170] Ollama 호출 완료: output_length=563


[최종 한국어 답변]
가장 최근 발생한 에러는 "배기/압력 계통 이상"으로, 이는 챔버 또는 배기 라인의 압력 조건이 기준 범위를 벗어나 작업이 중단되거나 보호 동작이 발생한 상황을 의미합니다.

### 주요 증상
- 공정 시작 직후 압력 안정화 지연
- 배기 밸브 응답 지연 또는 진공 도달 실패
- 설비 보호 정지 후 재시작 필요

### 가능한 원인
- 배기 라인 막힘 또는 밸브 오동작
- 압력 센서 노이즈 또는 보정값 이탈
- 레시피 전환 직후 초기화 미완료

### 영향 범위
- 로딩/언로딩 지연, 사이클 타임 증가, 특정 구간 반복 에러 가능성

### 우선 점검 사항
1. 축 위치 편차 로그와 서보 알람 이력 확인
2. 포토/리미트 센서 상태 및 오염 여부 확인
3. 최근 부품 교체 또는 티칭값 변경 이력 확인

### 권장 조치
1. 축 원점 복귀 및 기준 위치 재티칭
2. 센서 청소 및 케이블 체결 상태 확인
3. 반복 시 기구부 정렬 및 모터/드라이브 점검 요청

위 정보를 바탕으로 문제 해결을 진행하시면 됩니다. 추가적인 세부 정보가 필요하다면, 관련 로그와 이력 데이터를 참조해 보시기 바랍니다.


In [14]:
summary_payload = {
    "executed_steps": executed_steps,
    "query_type": query_type,
    "generated_sql": generated_sql,
    "has_db_result_summary": db_result_summary is not None,
    "rag_docs_count": len(rag_docs),
    "has_final_answer": final_answer is not None and str(final_answer).strip() != "",
    "errors_count": len(errors),
    "errors": errors,
}

print("[전체 실행 결과 요약]")
pprint(summary_payload)

display(
    pd.DataFrame(
        [
            {
                "query_type": query_type,
                "generated_sql": generated_sql,
                "db_summary": db_result_summary is not None,
                "rag_docs_count": len(rag_docs),
                "final_answer": final_answer,
            }
        ]
    )
)


[전체 실행 결과 요약]
{'errors': [],
 'errors_count': 0,
 'executed_steps': ['planner',
                    'sql_generation',
                    'sql_validation',
                    'db_execution_mock',
                    'db_summary',
                    'doc_rag_retrieval',
                    'final_answer'],
 'generated_sql': '```sql\n'
                  'SELECT TOP 1 EQPID, MODULEID, EVENTTIME, UNIT, STEP, '
                  'GLASSID, GLASSID2, LOTID, HOSTPPID, EQPPPID, ACTION, '
                  'ALARMID, ALARMTEXT, FILENAME, CREATETIME\n'
                  'FROM ERROR_LOG_DATA\n'
                  'ORDER BY EVENTTIME DESC\n'
                  '```',
 'has_db_result_summary': True,
 'has_final_answer': True,
 'query_type': 'DB_THEN_RAG',
 'rag_docs_count': 3}


,query_type,generated_sql,db_summary,rag_docs_count,final_answer
0,DB_THEN_RAG,"```sql\nSELECT TOP 1 EQPID, MODULEID, EVENTTIM...",True,3,"가장 최근 발생한 에러는 ""배기/압력 계통 이상""으로, 이는 챔버 또는 배기 라인의..."
